In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv("quote_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
quotes = df['quote']
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [6]:
quotes = quotes.str.lower()

In [7]:
quotes[0]

'“the world as we have created it is a process of our thinking. it cannot be changed without changing our thinking.”'

In [8]:
import string

# Create a translator to remove standard punctuation
translator = str.maketrans('', '', string.punctuation)

# Remove standard punctuation
quotes = quotes.apply(
    lambda x: x.translate(translator)
)

# Remove curly quotation marks and extra spaces
quotes = quotes.apply(
    lambda x: x.replace('“', '')
              .replace('”', '')
              .strip()
)

# Display first 5 quotes
quotes.head()

0    the world as we have created it is a process o...
1    it is our choices harry that show what we trul...
2    there are only two ways to live your life one ...
3    the person be it gentleman or lady who has not...
4    imperfection is beauty madness is genius and i...
Name: quote, dtype: object

In [9]:
quotes.head()

0    the world as we have created it is a process o...
1    it is our choices harry that show what we trul...
2    there are only two ways to live your life one ...
3    the person be it gentleman or lady who has not...
4    imperfection is beauty madness is genius and i...
Name: quote, dtype: object

In [10]:
quotes[0]

'the world as we have created it is a process of our thinking it cannot be changed without changing our thinking'

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
vocab_size = 10000
tokenizer = Tokenizer(num_words = vocab_size)
tokenizer.fit_on_texts(quotes)

In [13]:
word_index = tokenizer.word_index
print(len(word_index))
print(list(word_index.items())[:50])# based on word frequency id is provided

8847
[('the', 1), ('you', 2), ('to', 3), ('and', 4), ('a', 5), ('i', 6), ('is', 7), ('of', 8), ('that', 9), ('it', 10), ('in', 11), ('be', 12), ('not', 13), ('are', 14), ('your', 15), ('have', 16), ('for', 17), ('but', 18), ('we', 19), ('if', 20), ('what', 21), ('all', 22), ('with', 23), ('love', 24), ('can', 25), ('my', 26), ('when', 27), ('will', 28), ('as', 29), ('do', 30), ('who', 31), ('or', 32), ('me', 33), ('he', 34), ('they', 35), ('life', 36), ('one', 37), ('was', 38), ('like', 39), ('there', 40), ('people', 41), ('on', 42), ('its', 43), ('at', 44), ('so', 45), ('never', 46), ('no', 47), ('them', 48), ('dont', 49), ('know', 50)]


In [14]:
sequence = tokenizer.texts_to_sequences(quotes)

In [15]:
for i in range(3):
    print(quotes[i])

the world as we have created it is a process of our thinking it cannot be changed without changing our thinking
it is our choices harry that show what we truly are far more than our abilities
there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle


In [16]:
for i in range(3):
    print(sequence[i])

[1, 62, 29, 19, 16, 947, 10, 7, 5, 1153, 8, 70, 284, 10, 146, 12, 810, 105, 751, 70, 284]
[10, 7, 70, 872, 373, 9, 433, 21, 19, 467, 14, 295, 52, 54, 70, 2450]
[40, 14, 53, 201, 716, 3, 81, 15, 36, 37, 7, 29, 322, 93, 7, 5, 1043, 1, 101, 7, 29, 322, 126, 7, 5, 1043]


In [17]:
print(tokenizer.word_index['the'])
print(tokenizer.word_index['world'])
print(tokenizer.word_index['as'])
print(tokenizer.word_index['we'])
print(tokenizer.word_index['it'])
print(tokenizer.word_index['is'])


1
62
29
19
10
7


In [18]:
X = []
y = []
for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [19]:
len(X)

85269

In [20]:
len(y)

85269

In [21]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen = max_len, padding = 'pre')

In [23]:
X_padded

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,   62],
       [   0,    0,    0, ...,    1,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1122],
       [   0,    0,    0, ...,   19, 1122,    3],
       [   0,    0,    0, ..., 1122,    3,  171]],
      shape=(85269, 745), dtype=int32)

In [24]:
y = np.array(y)

In [25]:
X_padded.shape

(85269, 745)

In [26]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes = vocab_size)

In [28]:
y_one_hot.shape

(85269, 10000)

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [31]:
embedding_dim = 50
rnn_units = 128

In [33]:
#RNN
rnn_model = Sequential()
rnn_model.add(
    Embedding(input_dim = vocab_size, output_dim = embedding_dim,
              input_length = max_len))
rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units = vocab_size, activation = "softmax"))
    

C:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [34]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [35]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn (SimpleRNN)               │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# LSTM(Long Short Term Memory)

In [38]:
lstm_model = Sequential()
lstm_model.add(
        Embedding(input_dim = vocab_size, output_dim = embedding_dim, input_length = max_len))
lstm_model.add(LSTM(units = rnn_units))
lstm_model.add(Dense(units = vocab_size, activation = "softmax"))
    

C:\Users\Admin\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [39]:
#LSTM (Long Short Term Memory)
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']

)

In [40]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [42]:

#history_rnn = rnn_model.fit(
 #  epochs = 10,
  #  batch_size = 128,
   # validation_split = 0.1,
#verbose = 1
#)

In [46]:
#history_lstm = lstm_model.fit(
#    X_padded,
 #   y_one_hot,
  #  epochs = 100,
   # batch_size = 128,
    #validation_split = 0.1,
#verbose = 1
#)

In [47]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.21.0
GPUs: []
